# Lab 10 — Data Analysis & Aggregation
AI Detection Papers Analytics Pipeline

## Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

CLEANED_CSV = '../data/processed/cleaned/cleaned_data.csv'
df = pd.read_csv(CLEANED_CSV)
print(f'Loaded {len(df)} rows × {len(df.columns)} columns')
df.head()

## Part 1 — MySQL: Populate and Query paper_metrics

In [ ]:
from src.analytics.db_connector import run_db_pipeline

try:
    db_df = run_db_pipeline(df)
    print(f'MySQL round-trip OK: {len(db_df)} rows')
    display(db_df.head())
except Exception as e:
    print(f'MySQL unavailable: {e}')
    print('Using cleaned CSV as fallback for DB data')
    db_df = df[['paper_id','title','citation_count','relevance_score',
                'published_year','primary_category']].copy()
    db_df['paper_id'] = db_df['paper_id'].astype(str)

## Part 2 — Combining Data Sources (4 join types)

In [ ]:
from src.analytics.data_combiner import compare_join_types, combine_sources, save_join_comparison_chart
from pathlib import Path

csv_df = df.copy()
csv_df['paper_id'] = csv_df['paper_id'].astype(str)

counts = compare_join_types(csv_df, db_df)
print('Row counts per join type:')
for how, n in counts.items():
    print(f'  {how:6s}: {n} rows')

save_join_comparison_chart(counts, Path('../data/processed/analytics'))
print('\nBest join: INNER — only papers present in both sources')

combined_df = combine_sources(csv_df, db_df)
print(f'Combined DataFrame: {combined_df.shape}')

## Part 3 — Reshaping: melt() and pivot_table()

In [ ]:
from src.analytics.pivot_builder import (
    extract_primary_category, wide_to_long, long_to_wide,
    build_pivot_table, build_crosstab, save_pivot_csv
)
from pathlib import Path

df = extract_primary_category(df)

# Wide → Long (melt 3 numeric columns)
long_df = wide_to_long(df)
print('Long format:')
display(long_df.head(9))

# Long → Wide (pivot back)
wide_df = long_to_wide(long_df)
print('Back to wide:')
display(wide_df.head())

# Pivot table: avg citations by year × category
pivot = build_pivot_table(df)
print('Pivot table (avg citations by year × category):')
display(pivot)
save_pivot_csv(pivot, out_dir=Path('../data/processed/analytics'))

# Crosstab: language × year
ct = build_crosstab(df)
print('Crosstab (language × year):')
display(ct)

## Part 4 — GroupBy Analysis

In [ ]:
from src.analytics.aggregator import (
    category_summary, yearly_trends, top_n_per_group,
    save_category_csv, save_yearly_csv, save_yearly_chart
)
from pathlib import Path

ADIR = Path('../data/processed/analytics')

# Category summary (4 agg functions)
summary = category_summary(df)
print('Category summary:')
display(summary)
save_category_csv(summary, ADIR)

# Yearly trends (1990–2027)
trends = yearly_trends(df, start_year=2020, end_year=2027)
print('Yearly trends:')
display(trends)
save_yearly_csv(trends, ADIR)
save_yearly_chart(trends, ADIR)

# Top 3 papers per category
top3 = top_n_per_group(df, n=3)
print('Top 3 papers per category by citations:')
display(top3[['primary_category','title','citation_count']])

## Part 5 — Time Series Analysis

In [ ]:
from src.analytics.time_series import (
    parse_dates, extract_date_components,
    monthly_citation_series, resample_yearly,
    rolling_averages, save_rolling_chart
)
from pathlib import Path

df_ts = parse_dates(df)
df_ts = extract_date_components(df_ts)

valid   = df_ts['published_date'].notna().sum()
missing = df_ts['published_date'].isna().sum()
print(f'Valid dates: {valid} | Missing: {missing}')
print('Date components added:', ['ts_year','ts_month','ts_weekday','ts_quarter'])
display(df_ts[['title','published_date','ts_year','ts_month','ts_weekday','ts_quarter']].head())

monthly = monthly_citation_series(df_ts)
print(f'Monthly series: {len(monthly)} months')
display(monthly)

yearly = resample_yearly(monthly)
print('Yearly totals:')
display(yearly)

ra = rolling_averages(monthly)
print('Rolling averages (3/6/12-month):')
display(ra)

save_rolling_chart(ra, Path('../data/processed/analytics'))
print('Chart saved.')

## Part 6 — MongoDB Aggregation Pipeline

In [ ]:
from src.analytics.mongo_pipeline import build_aggregation_pipeline, run_mongo_pipeline

# Show the 4-stage pipeline definition
import json
pipeline = build_aggregation_pipeline(min_relevance=0.5)
print('Aggregation pipeline (4 stages):')
print(json.dumps(pipeline, indent=2))

# Run (uses pandas fallback if MongoDB is offline)
mongo_df = run_mongo_pipeline(min_relevance=0.5, df_fallback=df)
print(f'\nPipeline result ({len(mongo_df)} categories):')
display(mongo_df)

## Part 7 — Insight Reporter (4 Analytical Questions)

In [ ]:
from src.analytics.insight_reporter import run_all_questions
from pathlib import Path
import importlib, src.analytics.insight_reporter as ir

# Override output dir to relative path for notebook
ir.ANALYTICS_DIR = Path('../data/processed/analytics')

findings = run_all_questions(df)

# Show ROI chart
from IPython.display import Image
Image('../data/processed/analytics/category_roi.png')

## Part 8 — Full Pipeline Integration Check

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '../run_pipeline.py'],
    capture_output=True, text=True, cwd='..'
)
print('STDOUT:', result.stdout[-2000:] if result.stdout else '(none)')
print('STDERR:', result.stderr[-1000:] if result.stderr else '(none)')
print('Return code:', result.returncode)

## Analytics Output Files

In [ ]:
from pathlib import Path
analytics_dir = Path('../data/processed/analytics')
print('Files generated:')
for f in sorted(analytics_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} {size_kb:.1f} KB')